# 09 — Multimodal Fusion — Solution

---

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../..'))
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, matplotlib.pyplot as plt
from src.models.multimodal import PatchEmbedding, VisionEncoder, MultiModalFusion
from src.models.attention import MultiHeadAttention

torch.manual_seed(5)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## Part A — PatchEmbedding

In [ ]:
class MyPatchEmbedding(nn.Module):
    def __init__(self, image_size=224, patch_size=16, in_channels=3, d_model=768):
        super().__init__()
        assert image_size % patch_size == 0
        self.num_patches = (image_size // patch_size) ** 2
        self.patch_size = patch_size
        # Conv2d with non-overlapping patches — equivalent to linear projection of each patch
        self.projection = nn.Conv2d(in_channels, d_model, kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        x = self.projection(x)    # (B, d_model, H/P, W/P)
        x = x.flatten(2)          # (B, d_model, num_patches)
        x = x.transpose(1, 2)     # (B, num_patches, d_model)
        return x


patch_embed = MyPatchEmbedding(32, 4, d_model=64)
image = torch.randn(2, 3, 32, 32)
patches = patch_embed(image)
print(f'Patches: {patches.shape}   num_patches={patch_embed.num_patches}')

## Part B — Concatenation Fusion

In [ ]:
d_model = 64
patch_emb = PatchEmbedding(image_size=32, patch_size=8, d_model=d_model)
visual_tokens = patch_emb(torch.randn(2, 3, 32, 32))   # (2, 16, 64)

text_embed = nn.Embedding(100, d_model)
text_tokens = text_embed(torch.randint(0, 100, (2, 10)))  # (2, 10, 64)

# Concatenate along sequence dimension
multimodal_tokens = torch.cat([visual_tokens, text_tokens], dim=1)  # (2, 26, 64)

attn = MultiHeadAttention(d_model, n_heads=4)
out, _ = attn(multimodal_tokens, multimodal_tokens, multimodal_tokens)
print(f'Multimodal tokens : {multimodal_tokens.shape}')
print(f'Attention output  : {out.shape}')

## Part C — Cross-Modal Attention

In [ ]:
class CrossModalAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        self.attn = MultiHeadAttention(d_model, n_heads)

    def forward(self, text, image):
        # Q from text, K/V from image
        out, _ = self.attn(text, image, image)
        return out


cross = CrossModalAttention(64, 4)
result = cross(text_tokens, visual_tokens)
print(f'Cross-attention: {result.shape}   # text updated with visual context')

## Part D — Library Components

In [ ]:
vision_enc = VisionEncoder(image_size=32, patch_size=8, d_model=64, n_heads=4, n_layers=2)
image = torch.randn(2, 3, 32, 32)
visual_features = vision_enc(image)
print(f'VisionEncoder: {visual_features.shape}')

fusion = MultiModalFusion(d_model=64, n_heads=4)
fused  = fusion(text_tokens, visual_features)
print(f'Fused text   : {fused.shape}')

print('\nFusion approach comparison:')
strategies = [
    ('Token concat',    'Prepend img tokens to text',    'Simple, bidirectional', 'Long seqs'),
    ('Cross-attention', 'Text Q attends to img KV',      'Separate towers',      'More params'),
    ('Prefix tokens',   'Compressed visual repr',        'Short & fast',         'Lossy'),
]
for name, how, pro, con in strategies:
    print(f'  {name:<18}: {how:<32}  + {pro}  - {con}')